# Import libraries

In [5]:
import os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

load_dotenv()

True

# Initialize vectorstore and retrievers

In [3]:
api_key = os.getenv("API_KEY")
llm = model = GoogleGenerativeAI(
    api_key=api_key,
    model="gemini-2.5-flash-lite",
    temperature=0.0,
    max_tokens=50000,
    timeout=None,
    max_retries=2
)

collection_name = "langchain_docs_index"
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=api_key)

vectorstore = Chroma(embedding_function=embedding, collection_name=collection_name, persist_directory="./data/vectors/chroma_db")

In [7]:
# Pull all documents from the vectorstore
all_docs = vectorstore.get(include=['documents', 'metadatas'])

# Reconstruct Document objects for BM25
docs_from_chroma = [
    Document(page_content=text, metadata=meta) 
    for text, meta in zip(all_docs['documents'], all_docs['metadatas'])
]

bm25_retriever = BM25Retriever.from_documents(docs_from_chroma)
bm25_retriever.k = 2

In [8]:
len(docs_from_chroma)

6

In [9]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Ensemble retriever

In [10]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever], weights=[0.5, 0.5]
)

In [11]:
ensemble_retriever.invoke("What are Morphemes?")

[Document(metadata={'talks_about_tokenization': True, 'page': 1, 'importance_score': 'High', 'contains_regex': False, 'contains_code_or_cli': False, 'talks_about_evaluation_perplexity': False, 'talks_about_edit_distance': False, 'talks_about_unicode': False, 'content_type': 'Narrative', 'talks_about_bpe': False, 'source': 'data\\raw\\book_chapter_02.pdf', 'contains_table': False, 'chapter_number': 2, 'talks_about_smoothing_interpolation': False, 'section_title': 'Words', 'talks_about_ngrams': False, 'contains_math_latex': False, 'total_pages': 34, 'linguistic_focus': 'English', 'talks_about_morphology': False, 'talks_about_language_modeling': False, 'creationdate': 'D:20260329110007'}, page_content='2.1 • W ORDS 5\n2.1 Words\nHow many words are in the following sentence?\nThey picnicked by the pool, then lay back on the grass and\nlooked at the stars.\nThis sentence has 16 words if we don’t count punctuation as words, 18 if we\ncount punctuation. Whether we treat period (“ .”), comma (